# =========================================
# Day 2 - TF-IDF

- Term Frequency
- Inverse Document Frequency
- Why TF-IDF works
- Applications

# =========================================

---

# =========================================
# 1. INTUITION
# =========================================

Imagine you're reading a book and trying to find what makes a page "important."
- Words like "the", "is", "and" appear everywhere -> **not useful**
- Words like "transformer", "neura", "attention" -> **very infromative**

Now think:
- A word is important if:
    - It appears **frequently in a document**
    - But **not frequently across all documents**

That's exactly what TF-IDF does.

#### **What problem does this solve?**
- Bag of Words treats all words equally
- Common words dominate (noise)
- Important words get diluted

#### **Why was this needed?**
- Need to **weight words based on importance**
- Improve:
    - Search relevance
    - Text classification
    - Infromation retrieval

Key Idea:
**Important word = frequent in document + rare globally**

---

# =========================================
# 2. CORE CONCEPTS
# =========================================

#### **2.1 Term Frequency (TF)**

**Definition**
Measures how often word appears in a document

---

**Intuition**
- More occurences -> more importance (within document)

---

**Example**

Document: "AI is the future of AI"

- TF(AI) = 2
- TF(future) = 1

---

**Normalized TF (optional)**

TF = (word count) / (total words in document)

---

#### **2.2 Inverse Document Frequency (IDF)**

**Definition**
Measures how rare a word is across all documents

---

**Intuition**
- Rare words -> high importance
- Common words -> low importance

---

**Example**

Corpus:
- Doc1: "AI is amazing"
- Doc2: "AI is the future"

- "AI" appears in all docs -> low IDF
- "amazing" appears in one doc -> high IDF

---

**Conceptual Formula**

IDF(word) = log(total_docs / docs_containing_word)

---

#### **2.3 TF-IDF**

**Definition**
Combination of TF and IDF

---

**Idea**
TF-IDF = TF * IDF

---

**Meaning**
- High TF + High IDF -> very important word
- High TF + Low IDF -> common word (not important)

---

#### **2.4 Why TF-IDF Works**

- Penalizes common words
- Rewards rare, informative words
- Balances local importance **TF** + global importance **IDF**

---

#### **2.5 Step-by-Step Working**

1. Tokenize documents
2. Build vocabulary
3. Calculate TF for each word per document
4. Calculate IDF across corpus
5. Multiply TF * IDF
6. Create weighted vector

---

# =========================================
# 3. INTERVIEW NOTES
# =========================================

#### **Key Points**
- TF-IDF improves BoW by weighting words
- Combines local frequency + global rarity
- Widely used in search and retrieval systems

---

#### **Advantages**
- Reduces impact of common words
- Highlights meaningful words
- Simple and effective baseline

---

#### **Disadvantages**
- No context or semantics
- Still produces sparse vectors
- Cannot capture word relationships

---

#### **When to Use**

- Search engines
- Document ranking
- Text classification
- Keyword extraction

---

#### **Important Comparisons**

**BoW vs TF-IDF**

|**Feature** | **BoW** | **TF-IDF** |
|------------|---------|------------|
| Weighting | Raw count | Importance based |
| Performance | Basic | Better |

---

**TF-IDF vs Embeddings**

| **Feature** | **TF-IDF** | **Embeddings** |
|-------------|------------|----------------|
| Context | No | Yes |
| Representation | Sparse | Dense |

---

# =========================================
# 4. IMPLEMENTATION
# =========================================

#### **4.1 Using Sklearn**

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

coprus = [
    "AI is the future",
    "AI is powerful",
    "Machine learning is the future"
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(coprus)
print(X)
print("Vocabulary: ", vectorizer.get_feature_names_out())
print("TF-IDF Matrix:\n", X.toarray())

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 12 stored elements and shape (3, 7)>
  Coords	Values
  (0, 0)	0.5268201732399633
  (0, 2)	0.40912286076708654
  (0, 6)	0.5268201732399633
  (0, 1)	0.5268201732399633
  (1, 0)	0.5478321549274363
  (1, 2)	0.4254405389711991
  (1, 5)	0.7203334490549893
  (2, 2)	0.3154441510317797
  (2, 6)	0.4061917781433946
  (2, 1)	0.4061917781433946
  (2, 4)	0.5340933749435833
  (2, 3)	0.5340933749435833
Vocabulary:  ['ai' 'future' 'is' 'learning' 'machine' 'powerful' 'the']
TF-IDF Matrix:
 [[0.52682017 0.52682017 0.40912286 0.         0.         0.
  0.52682017]
 [0.54783215 0.         0.42544054 0.         0.         0.72033345
  0.        ]
 [0.         0.40619178 0.31544415 0.53409337 0.53409337 0.
  0.40619178]]


In [25]:
from sklearn.metrics.pairwise import cosine_similarity

query = ["machine learning"]
q_vec = vectorizer.transform(query)

similarities = cosine_similarity(q_vec, X)

print(similarities)

[[0.         0.         0.75532209]]


---

#### **4.2 Manual Implementation**

In [26]:
from collections import Counter
import math

def compute_tf(doc):
    words = doc.split()
    total = len(words)
    counts = Counter(words)

    return {word: count/total for word,count in counts.items()}

def compute_idf(docs):
    N = len(docs)
    idf = {}
    all_words = set(word for doc in docs for word in doc.split())

    for word in all_words:
        containing = sum(1 for doc in docs if word in doc.split())
        idf[word] =  math.log(N/ (1+containing))

    return idf

def compute_tfidf(docs):
    tfidf = []
    idf = compute_idf(docs)

    for doc in docs:
        tf = compute_tf(doc)
        # print(tf)
        tfidf.append({word: tf[word] * idf[word] for word in tf})

    return tfidf

docs = [
    "ai is the future",
    "ai is powerful",
    "machine learning is the future"
]

print(compute_tfidf(docs))


[{'ai': 0.0, 'is': -0.07192051811294523, 'the': 0.0, 'future': 0.0}, {'ai': 0.0, 'is': -0.09589402415059363, 'powerful': 0.13515503603605478}, {'machine': 0.08109302162163289, 'learning': 0.08109302162163289, 'is': -0.05753641449035618, 'the': 0.0, 'future': 0.0}]


---

#### **4.3 Variation with N-grams with TF-IDF**


In [27]:
vectorizer = TfidfVectorizer(ngram_range=(1,2))
X = vectorizer.fit_transform(coprus)

print(vectorizer.get_feature_names_out())

['ai' 'ai is' 'future' 'is' 'is powerful' 'is the' 'learning'
 'learning is' 'machine' 'machine learning' 'powerful' 'the' 'the future']


---

# =========================================
# 5. GENAI / LLM / AGENT MAPPING
# =========================================

#### **In LLMs**

- Not used directly
- Replaced by:
    - Embeddings
    - Attention mechanisms

---

#### **In RAG Systems**
- Used for:
    - Keyword based-based retrieval
    - Initial filtering before embeddings

---

#### **In AI Agents**
- Used in:
    - Document ranking
    - Search relevance
    - Query matching

---

#### **When NOT Needed**
- Semantic search with embeddings
- Deep learning NLP pipelines

---

# =========================================
# 6. MINI PROJECT / TASK
# =========================================

#### **Project: Document Ranking System**

**Input**

Query: "AI future"

Documents:[
    "AI is the future",
    "Machine learning is powerful",
    "Future of AI is bright"
]

---

**Expected Output**
- Ranked documents based on TF-IDF similarity

---

**Steps**

1. Convert documents to TF-IDF vectors
2. Convert query to TF-IDF vectors
3. Compute similarity(cosine similarity)
4. Rank documents

---

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

coprus = [
    "AI is the future",
    "Machine learning is powerful",
    "Future of AI is bright"
]


vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(coprus)

# print("Vocabulary: ", vectorizer.get_feature_names_out())
# print("Sparse vectors: \n", X.toarray())

query = ["AI future"]
v_query = vectorizer.transform(query)

print("similarities", cosine_similarity(v_query,X))

similarities [[0.67947078 0.         0.57444192]]


---

# =========================================
# 7. LEARNINGS / SUMMARY
# =========================================

#### **Key Takeaways**

- TF-IDF improves BoW by adding importance weighting
- Combines frequency + rarity
- Widely used in search systems

---

#### **Important Insights**

- Common words are less useful
- Rare words carry more information
- Still lacks semantic understanding

---

#### **Common Mistakes**

- Treating TF-IDF as semantic model
- Ignoring preprocessing
- Using in deep learning pipelines
- Not normalizing text

---

# =========================================
# 8. SELF-TEST QUESTIONS
# =========================================

1. **Why does TF-IDF down-weight common words?**

TF-IDF down weight common words because such words appear frequently across many documents and therefore carry little discriminative value. Words like "is", "the", and "and" ae syntactically necessary but do not help distinguish one document from another. The inverse document frequency (IDF) component assigns lower weights to those globally frequent terms and higher weights to rare term that appear in fewer documents. This ensures that information and domain specific words such as "transformers" or "neural network" have greater impact on the representation. As a result, TF-IDF improves tasks like search and document ranking by emphasizing meaningful keywords while supressing noise from common terms.

---

2. **How does TF-IDF improve over Bag of Words?**

TF-IDF imroves Bag of Words by introducing weighting mechanism that considers both local importance (term frequency withing a document) and global imortance (rarity across the corpus). While Bag of Words treats all words equally and relies purely on raw counts, TF-IDF reduces the influence of commonly occuring words and highlights more information ones. This leads to better document representation, especially for tasks like information retrieval and search, where distinguishing between documents is critical. Although both methods produce sparse vectors and lack semantic understanding, TF-IDF provides a more meaningful signal by prioritizing discriminative terms rather then just frequency.

---

3. **Why can TF-IDF fail in semantic understanding tasks?**

TF-IDF fails in semantic understanding tasks because it represents text as independent weighted words without capturing relationships, context or meaning between them. It does not account for word order, syntanx or semantic similarity meaning it cannot distinguish between phrases like "not good" and "good", or recognize that different words may have similar meanings (eg. "car" and "automobile"). Since it relies purely on statistical frequency rather than contextual understanding, TF-IDF is limited in tasks that require deeper language comprehension. In contrast, modern approaches like word embeddings and transfomer-based model capture semantic and contextual information, making them far more effective for such tasks.

---